# 🎯 Module 04: Classification

> **Stage:** Machine Learning  
> **Level:** Beginner → Intermediate  
> **Module:** 04 — Classification  
> **Date:** 08 January 2026  
> **Previous:** 03 — Regression  
> **Next:** 05 — Ensemble Methods

---

**Classification** is the family of supervised-learning models that predict **discrete categories**.

In this notebook we'll cover:

- What classification is
- **Binary** vs **multiclass** classification
- **Logistic Regression** (sigmoid-based)
- **K-Nearest Neighbors (KNN)**
- **Naive Bayes**
- **Support Vector Machines (SVM)**
- **Decision Trees**
- **Confusion matrix** and the four cells
- **Accuracy, Precision, Recall, F1**
- **ROC-AUC**
- Handling **imbalanced** data

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain what classification is (and how it differs from regression)
- Distinguish **binary** vs **multiclass** classification
- Use **LogisticRegression**, **KNN**, **Naive Bayes**, **SVM**, and **DecisionTreeClassifier**
- Explain the **sigmoid** function and the role of the **threshold**
- Read a **confusion matrix**
- Compute and interpret **accuracy, precision, recall, F1**
- Explain **ROC-AUC**
- Diagnose why **accuracy can mislead** on imbalanced data
- Choose the right **metric** for a business problem

---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve,
    classification_report,
)

np.random.seed(42)
print('✅ Setup ready.')

---
## 🧠 1. What is Classification?

**Classification** is a supervised ML task where the model predicts a **discrete class or category**.

| Problem | Classes |
| --- | --- |
| Email detection | Spam / Not Spam |
| Disease detection | Positive / Negative |
| Fraud detection | Fraud / Legitimate |
| Image classification | Cat / Dog |
| Customer churn | Churn / No Churn |

### Workflow

```text
Features (X)
     ↓
Classification Model
     ↓
Predicted Class
```

### Regression vs Classification

```text
Regression     → 25,000
Classification → 'Spam'
```

---
## 🔢 2. Binary vs Multiclass Classification

### Binary

Only two classes:

```text
Spam / Not Spam
Yes / No
0 / 1
```

### Multiclass

More than two classes:

```text
Cat
Dog
Horse
Bird
```

The model chooses **one class** from multiple possible classes.

---
## 📈 3. Logistic Regression

Despite its name, **Logistic Regression is a classification algorithm**.

It predicts a **probability between 0 and 1** using the **sigmoid** function:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

### Example

```text
Model output = 0.87
→ 87% probability of class 1
```

### Decision rule

```text
Probability >= 0.5 → Class 1
Probability <  0.5 → Class 0
```

> The threshold does **not** have to be 0.5. It can be tuned based on the cost of FP vs FN.

In [ ]:
# Visualize the sigmoid
z = np.linspace(-8, 8, 400)
sig = 1 / (1 + np.exp(-z))

plt.figure(figsize=(7, 3.5))
plt.plot(z, sig, linewidth=2)
plt.axhline(0.5, color='red', linestyle='--', label='threshold = 0.5')
plt.axhline(0, color='k', linewidth=0.5)
plt.axhline(1, color='k', linewidth=0.5)
plt.title('Sigmoid function σ(z) = 1 / (1 + e^{-z})')
plt.xlabel('z'); plt.ylabel('σ(z)')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Load the Breast Cancer dataset (binary classification)
data = load_breast_cancer()
X = data.data
y = data.target   # 0 = malignant, 1 = benign

print('Shape:', X.shape)
print('Classes:', data.target_names.tolist())
print('Class counts:', np.bincount(y).tolist())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('\nTrain:', X_train.shape, ' Test:', X_test.shape)

In [ ]:
# Logistic Regression — probabilities and predictions
log_reg = LogisticRegression(max_iter=200, random_state=42)
log_reg.fit(X_train, y_train)

preds = log_reg.predict(X_test)
probs = log_reg.predict_proba(X_test)[:, 1]

print('First 5 probabilities:', probs[:5].round(3))
print('First 5 predictions  :', preds[:5])
print(f'\nAccuracy: {accuracy_score(y_test, preds):.4f}')

---
## 👥 4. K-Nearest Neighbors (KNN)

KNN predicts a sample based on the **nearest training examples**.

### How it works

1. Choose `K`
2. Calculate distance to training samples
3. Find the `K` nearest samples
4. Take the **majority class**

```text
K = 5

Nearest neighbors:
Blue
Blue
Blue
Red
Red

Prediction → Blue
```

⚠️ KNN is distance-based, so **feature scaling is essential**.

In [ ]:
# KNN — MUST be scaled
knn_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5)),
])
knn_pipe.fit(X_train, y_train)
knn_preds = knn_pipe.predict(X_test)

print(f'KNN accuracy: {accuracy_score(y_test, knn_preds):.4f}')

---
## 🧮 5. Naive Bayes

Based on **Bayes' theorem**:

$$P(A|B) = \frac{P(B|A)\, P(A)}{P(B)}$$

**Naive assumption:** features are conditionally **independent** given the class.

Despite this simplification, Naive Bayes works very well for some tasks — especially **text classification**.

### Variants

| Variant | Use case |
| --- | --- |
| `GaussianNB` | Continuous numerical features |
| `MultinomialNB` | Counts / text features |
| `BernoulliNB` | Binary features |

In [ ]:
# Naive Bayes
nb = GaussianNB()
nb.fit(X_train, y_train)
nb_preds = nb.predict(X_test)

print(f'Naive Bayes accuracy: {accuracy_score(y_test, nb_preds):.4f}')

---
## 📐 6. Support Vector Machine (SVM)

SVM finds the **decision boundary** that separates classes with the **largest margin**.

```text
🟢 🟢 🟢 | 🔴 🔴
🟢 🟢 🟢 | 🔴 🔴
🟢 🟢 🟢 | 🔴 🔴
          ↑
     Decision boundary
```

### Key concepts

- **Hyperplane** — the decision boundary
- **Margin** — the gap between classes and the hyperplane
- **Support vectors** — the samples that define the margin
- **Kernel** — allows non-linear boundaries

Popular kernels: `linear`, `rbf`, `poly`.

⚠️ SVM is **scale-sensitive** — always standardize features first.

In [ ]:
# SVM with RBF kernel (scaled)
svm_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', probability=True, random_state=42)),
])
svm_pipe.fit(X_train, y_train)
svm_preds = svm_pipe.predict(X_test)

print(f'SVM (RBF) accuracy: {accuracy_score(y_test, svm_preds):.4f}')

---
## 🌳 7. Decision Trees

A Decision Tree makes predictions using a **sequence of questions**.

```text
       Age > 30?
        /      \
      Yes       No
      /          \
Income > 50K?   No Churn
   /    \
 Yes     No
 |        |
Churn   No Churn
```

### Split criteria

- **Gini impurity**
- **Entropy / information gain**

> Decision Trees **don't require feature scaling** — they split on raw values.

In [ ]:
# Decision Tree
tree = DecisionTreeClassifier(max_depth=5, random_state=42)
tree.fit(X_train, y_train)
tree_preds = tree.predict(X_test)

print(f'Decision Tree accuracy: {accuracy_score(y_test, tree_preds):.4f}')

---
## 📊 8. Confusion Matrix

A confusion matrix compares predictions with actual classes.

| | Predicted Positive | Predicted Negative |
| --- | ---: | ---: |
| **Actual Positive** | TP | FN |
| **Actual Negative** | FP | TN |

### Terms

- **TP** — predicted positive and actually positive
- **TN** — predicted negative and actually negative
- **FP** — predicted positive but actually negative
- **FN** — predicted negative but actually positive

### Example (disease detection)

```text
TP → Disease correctly detected
TN → Healthy person correctly identified
FP → Healthy person incorrectly flagged
FN → Disease missed
```

In [ ]:
# Confusion matrix for Logistic Regression
cm = confusion_matrix(y_test, preds)

print('Confusion matrix:')
print(f'              Predicted Neg  Predicted Pos')
print(f'Actual Neg       {cm[0,0]:>10}  {cm[0,1]:>13}')
print(f'Actual Pos       {cm[1,0]:>10}  {cm[1,1]:>13}')

In [ ]:
# Confusion matrix heatmap
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['Pred Neg', 'Pred Pos'])
ax.set_yticklabels(['Actual Neg', 'Actual Pos'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=14)
plt.colorbar(im)
plt.title('Confusion Matrix — Logistic Regression')
plt.tight_layout(); plt.show()

---
## 🎯 9. Classification Metrics

### Accuracy

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

> ⚠️ Misleading with imbalanced data.

### Precision

$$\text{Precision} = \frac{TP}{TP + FP}$$

Of everything predicted positive, how much was actually positive?

Useful when **false positives are costly**.

### Recall

$$\text{Recall} = \frac{TP}{TP + FN}$$

Of all actual positive cases, how many did the model find?

Useful when **false negatives are costly**.

### F1 Score

$$F1 = 2 \cdot \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

Balance between precision and recall.

In [ ]:
# All metrics in one place
print('Classification Report — Logistic Regression:')
print(classification_report(y_test, preds, target_names=data.target_names))

In [ ]:
# Compare metrics across models
models = {
    'Logistic':      (log_reg, X_test),
    'KNN (scaled)':  (knn_pipe, X_test),
    'Naive Bayes':   (nb, X_test),
    'SVM (scaled)':  (svm_pipe, X_test),
    'Decision Tree': (tree, X_test),
}

print(f'{"Model":<15} {"Acc":>7} {"Prec":>7} {"Rec":>7} {"F1":>7}')
print('-' * 48)
for name, (m, Xt) in models.items():
    p = m.predict(Xt)
    print(f'{name:<15} '
          f'{accuracy_score(y_test, p):>7.3f} '
          f'{precision_score(y_test, p):>7.3f} '
          f'{recall_score(y_test, p):>7.3f} '
          f'{f1_score(y_test, p):>7.3f}')

---
## 📈 10. ROC-AUC

Classifiers can produce **probabilities**, and the ROC curve examines:

```text
True Positive Rate
        vs
False Positive Rate
```

**AUC** = Area Under the ROC Curve.

```text
AUC = 1.0 → Perfect separation
AUC ≈ 0.5 → Random-like discrimination
```

For models without `predict_proba()`, use `decision_function()`.

In [ ]:
# ROC-AUC for Logistic Regression
probs = log_reg.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, probs)
print(f'ROC-AUC (Logistic Regression): {auc:.4f}')

# Plot the ROC curve
fpr, tpr, _ = roc_curve(y_test, probs)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, linewidth=2, label=f'LogReg (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.5)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## ⚠️ 11. Imbalanced Classification

Suppose:

```text
1000 transactions

990 → Legitimate
10  → Fraud
```

A model predicting **everything as legitimate** gets:

```text
Accuracy = 99%
```

But it detects:

```text
Fraud = 0 / 10
```

> **Accuracy alone is misleading.**

### Better metrics for imbalanced data

- Precision
- Recall
- F1
- Precision-Recall curve
- Confusion matrix
- ROC-AUC

In [ ]:
# Simulate class imbalance
X_imb, y_imb = make_classification(
    n_samples=1000, n_features=10,
    weights=[0.99, 0.01], flip_y=0,
    random_state=42,
)

print('Class distribution:', np.bincount(y_imb).tolist())

X_tr, X_te, y_tr, y_te = train_test_split(
    X_imb, y_imb, test_size=0.2, random_state=42, stratify=y_imb
)

# Dummy classifier: predict majority class only
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy='most_frequent').fit(X_tr, y_tr)
dummy_preds = dummy.predict(X_te)

print('\nDummy (always majority) classifier:')
print(f'  Accuracy: {accuracy_score(y_te, dummy_preds):.4f}   ← looks great!')
print(f'  Recall  : {recall_score(y_te, dummy_preds):.4f}   ← but useless')
print(f'  F1      : {f1_score(y_te, dummy_preds):.4f}')

In [ ]:
# A real classifier on the imbalanced data
lr_imb = LogisticRegression(max_iter=200, class_weight='balanced', random_state=42)
lr_imb.fit(X_tr, y_tr)
preds_imb = lr_imb.predict(X_te)

print('Logistic Regression (class_weight="balanced"):')
print(f'  Accuracy: {accuracy_score(y_te, preds_imb):.4f}')
print(f'  Recall  : {recall_score(y_te, preds_imb):.4f}   ← detects the minority class')
print(f'  F1      : {f1_score(y_te, preds_imb):.4f}')

---
## 🎚️ 12. Threshold Tuning

The default threshold is **0.5**, but you can move it based on business cost.

```text
Lower threshold → more positive predictions → higher recall, lower precision
Higher threshold → fewer positive predictions → higher precision, lower recall
```

In [ ]:
# Threshold tuning on the breast cancer model
probs = log_reg.predict_proba(X_test)[:, 1]

print(f'{"Threshold":>10} {"Precision":>10} {"Recall":>10} {"F1":>10}')
print('-' * 46)
for t in [0.2, 0.3, 0.5, 0.7, 0.8]:
    preds_t = (probs >= t).astype(int)
    print(f'{t:>10.1f} '
          f'{precision_score(y_test, preds_t):>10.3f} '
          f'{recall_score(y_test, preds_t):>10.3f} '
          f'{f1_score(y_test, preds_t):>10.3f}')

---
## 🧪 13. Basic Classification Workflow

```text
Dataset
   ↓
Preprocessing
   ↓
Train/Test Split
   ↓
Train Classifier
   ↓
Predict
   ↓
Confusion Matrix
   ↓
Precision / Recall / F1
   ↓
ROC-AUC
```

---
## 📝 14. Practice Checklist

Build a classification model using a dataset such as **Breast Cancer**, **Titanic**, or **SMS Spam**.

Try each of these classifiers:

- [ ] Logistic Regression
- [ ] KNN
- [ ] Naive Bayes
- [ ] SVM
- [ ] Decision Tree

Then compare:

- [ ] Accuracy
- [ ] Precision
- [ ] Recall
- [ ] F1
- [ ] ROC-AUC
- [ ] Confusion matrix

Also test what happens when you change the **classification threshold**.

---
## 🏋️ 15. Hands-On Exercises

### Exercise 1 — KNN Neighbor Sweep
Try `n_neighbors ∈ {1, 3, 5, 11, 25, 51}`. Plot accuracy vs K. Where does it peak? Where does it overfit or underfit?

### Exercise 2 — SVM Kernels
Compare `kernel ∈ {'linear', 'rbf', 'poly'}`. Report accuracy for each.

### Exercise 3 — Decision Tree Depth
Try `max_depth ∈ {1, 2, 5, 10, None}`. Which depth generalizes best? Watch the train/test gap.

### Exercise 4 — Multiclass Classification
Use `load_iris()` or `load_digits()`. Try each classifier. Report **macro F1**.

### Exercise 5 — Class Imbalance
Build an imbalanced dataset. Compare a plain classifier vs `class_weight='balanced'`. Which metric improves most?

### Exercise 6 — Confusion Matrix Reading
Given `TP=80, FP=15, FN=5, TN=100`, compute:
- Accuracy
- Precision
- Recall
- F1

Which metric would you prioritize if **false positives are costly**? What if **false negatives are costly**?

---
## 🎤 16. Interview Questions

1. **What is classification?**  
   Predicting a discrete class or category.

2. **Binary vs multiclass classification?**  
   Binary → two classes; multiclass → more than two classes.

3. **Why is Logistic Regression called "regression"?**  
   Because it models log-odds as a linear combination, but the output is a probability used for classification.

4. **What is the sigmoid function?**  
   $\sigma(z) = 1/(1+e^{-z})$ — squashes any real number into (0, 1).

5. **How does KNN work?**  
   It classifies by majority vote among the K nearest training samples.

6. **Why does KNN require scaling?**  
   Because it uses distances — unscaled features dominate the distance.

7. **What is Bayes' theorem?**  
   $P(A|B) = P(B|A)P(A)/P(B)$ — the foundation of Naive Bayes.

8. **Main assumption of Naive Bayes?**  
   Features are conditionally independent given the class.

9. **How does SVM classify data?**  
   It finds a maximum-margin hyperplane separating classes; kernels extend this to non-linear cases.

10. **What are support vectors?**  
    The training samples that lie closest to the decision boundary and define it.

11. **How does a Decision Tree make splits?**  
    It chooses feature/threshold combinations that minimize impurity (Gini) or maximize information gain.

12. **What are TP, TN, FP, FN?**  
    True Positive, True Negative, False Positive, False Negative.

13. **Precision vs Recall?**  
    Precision = TP/(TP+FP) — how trustworthy positives are. Recall = TP/(TP+FN) — how many positives were found.

14. **When is accuracy misleading?**  
    When classes are imbalanced — a model can be 99% accurate and still miss all positives.

15. **What is F1 score?**  
    Harmonic mean of precision and recall — balances both.

16. **What does ROC-AUC measure?**  
    The model's ability to rank positives higher than negatives across all thresholds.

---
## 🏁 17. Key Takeaways

```text
Classification → Predict categories

Logistic Regression → Probability-based classification
KNN               → Nearest neighbors
Naive Bayes       → Bayes theorem
SVM               → Maximum-margin boundary
Decision Tree     → Rule-based splits

Accuracy  → Overall correctness
Precision → Correctness of positive predictions
Recall    → Positive cases found
F1        → Precision + Recall balance
ROC-AUC   → Ranking / discrimination ability
```

### 🧭 Golden Rule

> **Pick the metric that matches the cost of your mistakes. On imbalanced data, accuracy is usually the wrong metric.**

---

### 🔗 What's Next?

**Module 05 — Ensemble Methods**

Single models have limits. We'll combine many weak learners into **strong ones** — Random Forest, Gradient Boosting, XGBoost, LightGBM, CatBoost, stacking, and voting.